In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('ggplot')
plt.rcParams['grid.color'] = 'white'
from scipy.optimize import curve_fit
import scipy.integrate as spi
import pandas as pd
import glob
import seaborn as sns
from kin_functions import *

# Different simulations

In [ ]:
# substrate info
# #               name,          NoC,    NoE
info_glu = [    'glucose',       6,      24]
info_ox  = [    'oxalate',       2,      2]

# total electrons in feed
e_tot_in = 0.5     # emol/L 

# kinetic parameters
K_glu = 10*10**(-6)    # M = 0.01 mM     
K_ox = K_glu

## Rewrite oxalate specialist Ks

In [ ]:
Y_ox_an_tot = -2.097115154 #mol/Cmol X
Y_ox_met_tot = -4.221893491 #mol/Cmol X
Y_ox_cat_tot = Y_ox_met_tot - Y_ox_an_tot #mol/Cmol/X
mu_max_ox = 0.464234079720467
q_s_cat_max_ox = -2 / info_ox[-1]        # mol S/Cmol X/h
q_s_max_ox = mu_max_ox * Y_ox_met_tot          # mol S/Cmol X/h

Kc_fit_eD2, Ke_fit_eD2, _ = fit_kinetics(q_s_max_ox, K_ox, Y_ox_an_tot, Y_ox_cat_tot, q_s_cat_max_ox, mu_max_ox, info_ox[0], info_ox[-1])

## $f_{eD1}$ = $f_{feed}$

### µ > µ$_{max,oxalotrophs}$ and f $_{eD1}$ < opt

In [ ]:
# specify the substrate fraction in metabolism, how much of the total electrons come from eD1 
f_eD1_i = 0.67     # emol eD1/emol tot

# specify the substrate fraction in the feed
f_feed = f_eD1_i

# operational parameters: dilution rate (= growth rate), hydraulic retention time (HRT) and total substrate fed
D        = 0.6      # 1/h
HRT      = 1/D      # h

# run simulation
sol_1, kin_figs_1 = simulate_competition("info_summary_glu_ox.xlsx", info_glu, info_ox, K_glu, K_ox, f_eD1_i, f_feed, D, e_tot_in, 100)



### µ > µ$_{max,oxalotrophs}$ and f $_{eD1}$ = opt

In [ ]:
# specify the substrate fraction in metabolism, how much of the total electrons come from eD1 
f_eD1_i = 0.7778    # emol eD1/emol tot

# specify the substrate fraction in the feed
f_feed = f_eD1_i

# operational parameters: dilution rate (= growth rate), hydraulic retention time (HRT) and total substrate fed
D        = 0.6      # 1/h
HRT      = 1/D      # h

# run simulation
sol_2, kin_figs_2 = simulate_competition("info_summary_glu_ox.xlsx",info_glu, info_ox, K_glu, K_ox, f_eD1_i, f_feed, D, e_tot_in, 100)

#### The kinetic fit

In [ ]:
for fig in kin_figs_2:
    display(fig)

### µ > µ$_{max,oxalotrophs}$ and f $_{eD1}$ > opt 

In [ ]:
# specify the substrate fraction in metabolism, how much of the total electrons come from eD1 
f_eD1_i = 0.8572     # emol eD1/emol tot

# specify the substrate fraction in the feed
f_feed = f_eD1_i

# operational parameters: dilution rate (= growth rate), hydraulic retention time (HRT) and total substrate fed
D        = 0.6      # 1/h
HRT      = 1/D      # h

# run simulation
sol_3, kin_figs_3 = simulate_competition("info_summary_glu_ox.xlsx",info_glu, info_ox, K_glu, K_ox, f_eD1_i, f_feed, D, e_tot_in, 100)


### µ < µ$_{max,oxalotrophs}$ and f $_{eD1}$ < opt

In [ ]:
# specify the substrate fraction in metabolism, how much of the total electrons come from eD1 
f_eD1_i = 0.67     # emol eD1/emol tot

# specify the substrate fraction in the feed
f_feed = f_eD1_i

# operational parameters: dilution rate (= growth rate), hydraulic retention time (HRT) and total substrate fed
D        = 0.1      # 1/h
HRT      = 1/D      # h

# run simulation
sol_4, kin_figs_4 = simulate_competition("info_summary_glu_ox.xlsx",info_glu, info_ox, K_glu, K_ox, f_eD1_i, f_feed, D, e_tot_in, 100)


### µ < µ$_{max,oxalotrophs}$ and f $_{eD1}$ = opt

In [ ]:
# specify the substrate fraction in metabolism, how much of the total electrons come from eD1 
f_eD1_i = 0.7778     # emol eD1/emol tot

# specify the substrate fraction in the feed
f_feed = f_eD1_i

# operational parameters: dilution rate (= growth rate), hydraulic retention time (HRT) and total substrate fed
D        = 0.1      # 1/h
HRT      = 1/D      # h

# run simulation
sol_5, kin_figs_5 = simulate_competition("info_summary_glu_ox.xlsx",info_glu, info_ox, K_glu, K_ox, f_eD1_i, f_feed, D, e_tot_in, 100)

### µ < µ$_{max,oxalotrophs}$ and f $_{eD1}$ > opt

In [ ]:
# specify the substrate fraction in metabolism, how much of the total electrons come from eD1 
f_eD1_i = 0.8572     # emol eD1/emol tot

# specify the substrate fraction in the feed
f_feed = f_eD1_i

# operational parameters: dilution rate (= growth rate), hydraulic retention time (HRT) and total substrate fed
D        = 0.1      # 1/h
HRT      = 1/D      # h

# run simulation
sol_6, kin_figs_6 = simulate_competition("info_summary_glu_ox.xlsx",info_glu, info_ox, K_glu, K_ox, f_eD1_i, f_feed, D, e_tot_in, 100)

## VaryingK1/K2

In [ ]:
def survives_species3(c_X3, n_last=100):
    """
    Species 3 survives if:
      1) final biomass > initial biomass
      2) slope of last n_last points >= 0
    """
    # Condition 1: net positive change
    if c_X3[-1] <= c_X3[0]:
        return False
    
    # Extract last N points
    if len(c_X3) < n_last:
        n_last = len(c_X3) // 2  # fallback
    
    y = c_X3[-n_last:]
    x = np.arange(n_last)

    # Fit slope
    slope, _ = np.polyfit(x, y, 1)

    # Condition 2: slope must not be negative
    return slope >= 0

# Parameter grid
f_eD1_vals = np.arange(0.1, 1.0, 0.05)   # 0.0, 0.1, …, 1.0
rK1_K2_vals = np.logspace(-2, 2, 5)               # biologically relevant range, considering that K_glu is the reference value


In [ ]:
survival_matrix_spec_var_lowD = np.zeros((len(f_eD1_vals), len(rK1_K2_vals)), dtype=int)

D = 0.1
HRT_default = 30   # long enough for quasi-steady state

for i_f, f_eD1 in enumerate(f_eD1_vals):
    for i_k, rK1_K2_i in enumerate(rK1_K2_vals):
        print(f'{f_eD1} and Ks ratio = {rK1_K2_i}')

        K_ox = K_glu / rK1_K2_i

        sol, _ = simulate_competition("info_summary_glu_ox.xlsx",
            info_glu, info_ox,
            K_glu, K_ox,
            f_eD1, f_eD1,
            D,
            e_tot_in,
            HRT_default,
            dt=0.01)#, comp_plots=False)

        # species concentrations
        _, _, _, _, c_X3 = sol.y

        # Only track species 3 survival
        survival_matrix_spec_var_lowD[i_f, i_k] = int(survives_species3(c_X3))


In [ ]:
survival_matrix_spec_var_highD = np.zeros((len(f_eD1_vals), len(rK1_K2_vals)), dtype=int)

D = 0.6
HRT_default = 30   # long enough for quasi-steady state

for i_f, f_eD1 in enumerate(f_eD1_vals):
    for i_k, rK1_K2_i in enumerate(rK1_K2_vals):
        print(f'{f_eD1} and Ks ratio = {rK1_K2_i}')

        K_ox = K_glu / rK1_K2_i

        sol, _ = simulate_competition("info_summary_glu_ox.xlsx",
            info_glu, info_ox,
            K_glu, K_ox,
            f_eD1, f_eD1,
            D,
            e_tot_in,
            HRT_default,
            dt=0.01)#, comp_plots=False)

        # species concentrations
        _, _, _, _, c_X3 = sol.y

        # Only track species 3 survival
        survival_matrix_spec_var_highD[i_f, i_k] = int(survives_species3(c_X3))


## When generalist is fitted to a _different_ glucose specialist

In [ ]:
survival_matrix_glu_gen_lowD = np.zeros((len(f_eD1_vals), len(rK1_K2_vals)), dtype=int)

D = 0.1
HRT_default = 30   # long enough for quasi-steady state

for i_f, f_eD1 in enumerate(f_eD1_vals):
    for i_k, rK1_K2_i in enumerate(rK1_K2_vals):
        print(f'{f_eD1} and Ks ratio = {rK1_K2_i}')

        K_ox = K_glu 
        K_glu_gen_i = K_glu / rK1_K2_i

        sol, _ = simulate_competition("info_summary_glu_ox.xlsx",
            info_glu, info_ox,
            K_glu, K_ox,
            f_eD1, f_eD1,
            D,
            e_tot_in,
            HRT_default,
            dt=0.01, K_eD1_gen = K_glu_gen_i)#, comp_plots=False)

        # species concentrations
        _, _, _, _, c_X3 = sol.y

        # Only track species 3 survival
        survival_matrix_glu_gen_lowD[i_f, i_k] = int(survives_species3(c_X3))


In [ ]:
survival_matrix_glu_gen_highD = np.zeros((len(f_eD1_vals), len(rK1_K2_vals)), dtype=int)

D = 0.6
HRT_default = 30   # long enough for quasi-steady state

for i_f, f_eD1 in enumerate(f_eD1_vals):
    for i_k, rK1_K2_i in enumerate(rK1_K2_vals):
        print(f'{f_eD1} and Ks ratio = {rK1_K2_i}')

        K_ox = K_glu 
        K_glu_gen_i = K_glu / rK1_K2_i

        sol, _ = simulate_competition("info_summary_glu_ox.xlsx",
            info_glu, info_ox,
            K_glu, K_ox,
            f_eD1, f_eD1,
            D,
            e_tot_in,
            HRT_default,
            dt=0.01, K_eD1_gen = K_glu_gen_i)#, comp_plots=False)

        # species concentrations
        _, _, _, _, c_X3 = sol.y

        # Only track species 3 survival
        survival_matrix_glu_gen_highD[i_f, i_k] = int(survives_species3(c_X3))

In [ ]:
np.save("glu_ox_high_D_var_gen.npy", survival_matrix_glu_gen_highD)
np.save("glu_ox_high_D_var_spec.npy", survival_matrix_spec_var_highD)
np.save("glu_ox_low_D_var_gen.npy", survival_matrix_glu_gen_lowD)
np.save("glu_ox_low_D_var_spec.npy", survival_matrix_spec_var_lowD)

### Manually check 'odd' results of sensitivity analysis and adjust outcome if false negative

In [ ]:
glu_ox_high_D_var_gen      = np.load("glu_ox_high_D_var_gen.npy")
glu_ox_high_D_var_spec     = np.load("glu_ox_high_D_var_spec.npy")
glu_ox_low_D_var_gen       = np.load("glu_ox_low_D_var_gen.npy")
glu_ox_low_D_var_spec      = np.load("glu_ox_low_D_var_spec.npy")

In [ ]:
# for D = 0.1/h, there was originally no survival of the generalist plotted at f_eD1 = 0.8 
# at Kglu_spec/Kglu_gen = 100
# rerun
f_eD1_i = 0.80
f_feed = f_eD1_i

sol_1, kin_figs_1 = simulate_competition("info_summary_glu_ox.xlsx", info_glu, info_ox, K_glu, K_ox, f_eD1_i, f_feed, 0.1, e_tot_in, 100, K_eD1_gen = K_glu/100)


In [ ]:
# update outcome to 1: generalist survival
glu_ox_low_D_var_gen[np.where(np.isclose(f_eD1_vals, 0.8))[0], np.where(np.isclose(rK1_K2_vals, 100))[0]] = 1


In [ ]:
# save outcome matrices again with change(s)
np.save("glu_ox_high_D_var_gen.npy", glu_ox_high_D_var_gen)
np.save("glu_ox_high_D_var_spec.npy", glu_ox_high_D_var_spec)
np.save("glu_ox_low_D_var_gen.npy", glu_ox_low_D_var_gen)
np.save("glu_ox_low_D_var_spec.npy", glu_ox_low_D_var_spec)